# 05 Overfitting and Generalization

In [ ]:
# User-editable papermill/environment configuration
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/tmp/wwpgd_v2/data"))
RESULTS_ROOT = Path(
    os.environ.get(
        "WWGPT_RESULTS_ROOT",
        os.environ.get(
            "RESULTS_ROOT",
            "/tmp/wwpgd_v2/real_level0_results_v5/"
            "experiments/level_00/multiplier_20",
        ),
    )
)
RUN_LOG = Path(os.environ.get("RUN_LOG", "/tmp/wwpgd_v2/real_level0_five_seed_v4.log"))
PID_FILE = Path(os.environ.get("PID_FILE", "/tmp/wwpgd_v2/real_level0_five_seed_v4.pid"))

ANALYSIS_DIR = RESULTS_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_LEGACY = False
EXPECTED_SEEDS = {1337, 2027, 4099, 7919, 104729}


In [ ]:
import sys
from pathlib import Path
cwd = Path.cwd().resolve()
repo = cwd if (cwd/'src'/'wwgpt').exists() else cwd.parent
if str(repo/'src') not in sys.path:
    sys.path.insert(0, str(repo/'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from wwgpt.analysis import *
RESULTS_ROOT = resolve_experiment_root(RESULTS_ROOT)
ANALYSIS_DIR = RESULTS_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Run log: {RUN_LOG}')
print(f'PID file: {PID_FILE}')


Generalization and fixed-probe metrics for canonical schema-v2 pairs.

In [ ]:
candidates = discover_pair_candidates(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
canonical_pairs, pair_audit = select_canonical_pairs(candidates)
runs = discover_canonical_runs(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
inventory = build_run_inventory(runs)
pair_audit.to_csv(ANALYSIS_DIR/'pair_candidate_audit.csv', index=False)
inventory.to_csv(ANALYSIS_DIR/'run_inventory.csv', index=False)
print(f'Canonical pairs: {len(canonical_pairs)}')
display(pair_audit)
display(inventory)


In [ ]:
def plot_metric(metric, filename, ylabel=None, final_frac=None):
    fig, ax = plt.subplots(figsize=(7,4))
    byfam={}
    for r in runs:
        m=load_run_artifacts(Path(r['run_dir']))['metrics']
        if metric not in m or 'tokens_seen' not in m: continue
        if final_frac is not None:
            cutoff=m['tokens_seen'].max()*(1-final_frac); m=m[m['tokens_seen']>=cutoff]
        ax.plot(m['tokens_seen'], m[metric], alpha=.25, color=OPTIMIZER_COLORS[r['optimizer_family']])
        byfam.setdefault(r['optimizer_family'],[]).append(m)
    for fam,curves in byfam.items():
        grid, vals=align_curves(curves,'tokens_seen',metric)
        if len(grid):
            mean=vals.mean(axis=0); sem=vals.std(axis=0,ddof=1)/np.sqrt(vals.shape[0]) if vals.shape[0]>1 else 0
            t=stats.t.ppf(.975, vals.shape[0]-1) if vals.shape[0]>1 else 0
            ax.plot(grid,mean,lw=3,label=OPTIMIZER_LABELS[fam],color=OPTIMIZER_COLORS[fam])
            if vals.shape[0]>1: ax.fill_between(grid,mean-t*sem,mean+t*sem,alpha=.15,color=OPTIMIZER_COLORS[fam])
    ax.set_xlabel('tokens_seen'); ax.set_ylabel(ylabel or metric); ax.legend(); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/filename); plt.close(fig)

inventory.to_csv(ANALYSIS_DIR/'generalization_run_inventory.csv',index=False)
metrics=[]
for r in runs:
    m=load_run_artifacts(Path(r['run_dir']))['metrics']; m=add_generalization_measures(m, vocab_size=100); m=m.assign(seed=r['seed'],pair_id=r['pair_id'],optimizer_family=r['optimizer_family'],optimizer_label=r['optimizer_label']);
    if 'generalization_gap' in m: m['absolute_generalization_gap']=m['generalization_gap'].abs()
    metrics.append(m)
allm=pd.concat(metrics,ignore_index=True)
terms=[]
for metric in ['train_loss','validation_loss','generalization_gap','absolute_generalization_gap','train_perplexity','val_perplexity','perplexity_gap','perplexity_ratio','train_bits_per_token','val_bits_per_token','val_token_prediction_capacity','train_top1_accuracy','val_top1_accuracy','train_top5_accuracy','val_top5_accuracy','train_token_error','val_token_error']:
    if metric in allm:
        t=allm.sort_values('tokens_seen').groupby(['seed','pair_id','optimizer_family']).tail(1)[['seed','pair_id','optimizer_family',metric]]
        terms.append(t.rename(columns={metric:'value'}).assign(metric=metric))
terminal=pd.concat(terms,ignore_index=True) if terms else pd.DataFrame(); terminal.to_csv(ANALYSIS_DIR/'generalization_terminal_metrics.csv',index=False)
diffs=[]
for metric,g in terminal.groupby('metric'):
    p=g.pivot_table(index=['seed','pair_id'],columns='optimizer_family',values='value',aggfunc='first').dropna()
    if {'adamw','wwpgd'}.issubset(p.columns):
        p['wwpgd_minus_adamw']=p.wwpgd-p.adamw; p=p.reset_index(); p['metric']=metric; diffs.append(p)
(pd.concat(diffs,ignore_index=True) if diffs else pd.DataFrame()).to_csv(ANALYSIS_DIR/'paired_generalization_differences.csv',index=False)
for metric,file in [('val_perplexity','validation_perplexity_vs_tokens.png'),('val_token_prediction_capacity','token_prediction_capacity_vs_tokens.png'),('val_top1_accuracy','top1_accuracy_vs_tokens.png'),('generalization_gap','generalization_gap_vs_tokens.png'),('absolute_generalization_gap','absolute_generalization_gap_vs_tokens.png')]: plot_metric(metric,file,metric)
